# Diabetes Classification with CatBoost — Multi-Dataset Workflow

A reproducible CatBoost-only pipeline for diabetes classification across compatible health-indicator datasets. It supports **multiple CSV files**, validates schema and target encoding, reports dataset provenance, and trains one model using a harmonized feature set.

> **Research/education only:** this model does not diagnose diabetes. The BRFSS balanced dataset changes prevalence; its probabilities are not population-risk estimates. Combining datasets can introduce substantial differences in sampling, coding, and label definitions. Only combine datasets that have compatible definitions and units.

## Design decisions

- **Algorithm:** CatBoost is the sole classifier. It handles non-linear relationships and interactions well on tabular data and can work with missing numeric values. There is no universally best algorithm; performance must be measured on held-out data.
- **Multi-dataset support:** choose one of two modes:
  1. `combine_compatible_datasets=True`: concatenate compatible sources after schema/target checks.
  2. `False`: use the first discovered dataset, with an explicit `DATA_PATHS` list to control which one.
- **Avoid leakage:** split before threshold selection and final test evaluation. Dataset source is retained for auditing but excluded from predictors.
- **Schema mapping:** use `COLUMN_MAPS` to map source-specific column names into a common canonical schema.
- **Target:** canonical binary `Diabetes_binary` (`0` = no diabetes, `1` = diabetes). Set `TARGET_MAPS` when a source uses different target labels.

## 1. Install and import dependencies

In [ ]:
# In a fresh environment, uncomment if needed:
# %pip install catboost pandas numpy scikit-learn matplotlib seaborn joblib

from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix, RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from catboost import CatBoostClassifier
import joblib

RANDOM_STATE = 42
TARGET = "Diabetes_binary"
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

## 2. Configure datasets

Add one or more CSV paths. The notebook checks local paths and common Kaggle locations. You can use the CDC BRFSS 2015 health-indicators files and other datasets only after ensuring they share compatible target meaning and feature definitions.

**Important:** two files containing the same respondents or overlapping records should not be concatenated blindly. For robust evaluation across sources, consider holding out an entire source as an external test set.

In [ ]:
# Option A: explicitly list your dataset paths.
DATA_PATHS = [
    # "/kaggle/input/diabetes-health-indicators-dataset/diabetes_binary_health_indicators_BRFSS2015.csv",
    # "/kaggle/input/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv",
]

# If DATA_PATHS is empty, try these common paths.
AUTO_DISCOVER = True
combine_compatible_datasets = True

COMMON_PATHS = [
    Path("/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_binary_health_indicators_BRFSS2015.csv"),
    Path("/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv"),
    Path("/kaggle/input/diabetes-health-indicators-dataset/diabetes_binary_health_indicators_BRFSS2015.csv"),
    Path("/kaggle/input/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv"),
    Path("diabetes_binary_health_indicators_BRFSS2015.csv"),
    Path("diabetes_binary_5050split_health_indicators_BRFSS2015.csv"),
]

# Canonicalize column names across sources. Add mappings as needed.
# Example: {"diabetes": "Diabetes_binary", "BMI": "BMI"}
COLUMN_MAPS = {
    # "source_filename.csv": {"diabetes": "Diabetes_binary", "blood_pressure": "HighBP"}
}

# If a source target uses labels other than 0 and 1, map them explicitly.
# Example: {"source_filename.csv": {"No": 0, "Yes": 1}}
TARGET_MAPS = {
    # "source_filename.csv": {"No": 0, "Yes": 1}
}

# Set this only if you know all source features have equivalent definitions/units.
MIN_SHARED_FEATURES = 5

## 3. Load, normalize and validate datasets

In [ ]:
requested_paths = [Path(p) for p in DATA_PATHS if str(p).strip()]
if not requested_paths and AUTO_DISCOVER:
    requested_paths = [p for p in COMMON_PATHS if p.exists()]

if not requested_paths:
    raise FileNotFoundError(
        "No CSV dataset found. Add one or more CSV paths to DATA_PATHS."
    )

frames, dataset_report = [], []
for path in requested_paths:
    if not path.exists():
        warnings.warn(f"Skipping missing path: {path}")
        continue
    raw = pd.read_csv(path)
    raw.columns = raw.columns.astype(str).str.strip()
    mapping = COLUMN_MAPS.get(path.name, {})
    raw = raw.rename(columns=mapping)

    if TARGET not in raw.columns:
        warnings.warn(f"Skipping {path.name}: target column {TARGET!r} not found.")
        continue

    target_map = TARGET_MAPS.get(path.name)
    if target_map:
        raw[TARGET] = raw[TARGET].map(target_map)

    # Numeric health-indicator datasets are expected. Convert unsupported values to NaN.
    for col in raw.columns:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

    raw = raw.loc[raw[TARGET].isin([0, 1])].copy()
    if raw.empty:
        warnings.warn(f"Skipping {path.name}: no valid binary target rows.")
        continue
    raw[TARGET] = raw[TARGET].astype("int8")
    raw["__dataset_source__"] = path.name
    frames.append(raw)
    dataset_report.append({
        "source": path.name, "rows_after_target_validation": len(raw),
        "features_including_target": len(raw.columns) - 1,
        "positive_rate": float(raw[TARGET].mean())
    })

if not frames:
    raise ValueError("No usable datasets loaded. Check target names and TARGET_MAPS.")

display(pd.DataFrame(dataset_report))
print("Loaded sources:", len(frames))

## 4. Harmonize compatible feature sets

Only shared columns can safely be used in a single concatenated model unless missing columns have a well-defined meaning. This cell uses the intersection of feature names across loaded files and excludes the source label from predictors. It does **not** assume similarly named features have identical semantics; review the feature definitions before combining.

In [ ]:
if combine_compatible_datasets:
    shared = set(frames[0].columns)
    for frame in frames[1:]:
        shared &= set(frame.columns)
    shared.discard(TARGET)
    shared.discard("__dataset_source__")
    feature_columns = sorted(shared)
    if len(feature_columns) < MIN_SHARED_FEATURES:
        raise ValueError(
            f"Only {len(feature_columns)} shared features found; need at least "
            f"{MIN_SHARED_FEATURES}. Check COLUMN_MAPS or set combine_compatible_datasets=False."
        )
    aligned = []
    for frame in frames:
        piece = frame[feature_columns + [TARGET]].copy()
        piece["__dataset_source__"] = frame["__dataset_source__"].iloc[0]
        aligned.append(piece)
    data = pd.concat(aligned, ignore_index=True)
else:
    data = frames[0].copy()
    feature_columns = [c for c in data.columns if c not in [TARGET, "__dataset_source__"]]

# Remove rows with missing target (normally already handled); keep feature NaNs for CatBoost.
data = data.loc[data[TARGET].isin([0, 1])].copy()
X = data[feature_columns].copy()
y = data[TARGET].astype("int8").copy()
source = data["__dataset_source__"].copy()

print("Combined shape:", data.shape)
print("Features used:", len(feature_columns))
print("Sources:")
display(source.value_counts().to_frame("rows"))
print("Target prevalence by source:")
display(data.groupby("__dataset_source__")[TARGET].agg(["count", "mean"]).rename(columns={"mean":"positive_rate"}))
display(X.head())

## 5. Split into train, validation and test

For a simple pooled evaluation, we stratify by the target. If the goal is to generalize to a *new dataset*, use a source-held-out evaluation instead: train on one or more sources and evaluate on a completely unseen source. A random row split can look optimistic when datasets overlap.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15 / 0.85,
    random_state=RANDOM_STATE, stratify=y_trainval
)
print(f"Train={len(X_train):,}, validation={len(X_val):,}, test={len(X_test):,}")
print(f"Prevalence train/val/test: {y_train.mean():.3f} / {y_val.mean():.3f} / {y_test.mean():.3f}")

## 6. Train CatBoost only

CatBoost natively supports missing numeric values. Early stopping uses the validation set; no test information is used during training.

In [ ]:
cat_model = CatBoostClassifier(
    iterations=2500,
    depth=7,
    learning_rate=0.035,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    l2_leaf_reg=5.0,
    random_strength=0.5,
    auto_class_weights=None,  # set "Balanced" only when justified by the task
    verbose=200,
    allow_writing_files=False
)
cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=120,
    use_best_model=True,
    verbose=200
)
print("Best iteration:", cat_model.get_best_iteration())
print("Best validation score:", cat_model.get_best_score())

## 7. Select a decision threshold on validation data

The default threshold 0.5 is not necessarily optimal. This table makes the trade-off explicit. The example chooses the threshold with the highest validation F1; if the intended use is screening, a sensitivity target may be more appropriate and should be documented with clinical input.

In [ ]:
p_val = cat_model.predict_proba(X_val)[:, 1]
threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.025):
    pred = (p_val >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 3),
        "precision": precision_score(y_val, pred, zero_division=0),
        "sensitivity": recall_score(y_val, pred, zero_division=0),
        "specificity": recall_score(y_val, pred, pos_label=0, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_val, pred)
    })
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(3))
chosen_threshold = float(threshold_table.loc[threshold_table["f1"].idxmax(), "threshold"])
print(f"Validation-F1 threshold: {chosen_threshold:.3f}")

## 8. Final held-out test evaluation

In [ ]:
p_test = cat_model.predict_proba(X_test)[:, 1]
pred_test = (p_test >= chosen_threshold).astype(int)
metrics = {
    "model": "CatBoost",
    "best_iteration": int(cat_model.get_best_iteration()),
    "threshold": chosen_threshold,
    "test_accuracy": accuracy_score(y_test, pred_test),
    "test_balanced_accuracy": balanced_accuracy_score(y_test, pred_test),
    "test_precision": precision_score(y_test, pred_test, zero_division=0),
    "test_sensitivity": recall_score(y_test, pred_test, zero_division=0),
    "test_specificity": recall_score(y_test, pred_test, pos_label=0, zero_division=0),
    "test_f1": f1_score(y_test, pred_test, zero_division=0),
    "test_roc_auc": roc_auc_score(y_test, p_test),
    "test_average_precision": average_precision_score(y_test, p_test),
    "test_brier_score": brier_score_loss(y_test, p_test),
    "test_rows": int(len(y_test)),
    "test_prevalence": float(y_test.mean()),
    "dataset_sources": sorted(source.unique().tolist()),
    "features": feature_columns
}
display(pd.Series(metrics, name="value").to_frame())
print(classification_report(y_test, pred_test, target_names=["No diabetes", "Diabetes"], zero_division=0))

cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted no", "Predicted yes"],
            yticklabels=["Actual no", "Actual yes"])
plt.title(f"CatBoost test confusion matrix (threshold={chosen_threshold:.3f})")
plt.xlabel("Prediction"); plt.ylabel("Actual")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_test, p_test, ax=axes[0])
axes[0].set_title("ROC curve — test")
PrecisionRecallDisplay.from_predictions(y_test, p_test, ax=axes[1])
axes[1].set_title("Precision-recall curve — test")
plt.tight_layout(); plt.show()

## 9. Calibration and feature importance

In [ ]:
fraction_positive, mean_predicted = calibration_curve(
    y_test, p_test, n_bins=10, strategy="quantile"
)
plt.figure(figsize=(5, 5))
plt.plot(mean_predicted, fraction_positive, marker="o", label="CatBoost")
plt.plot([0, 1], [0, 1], "--", label="Ideal")
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed positive fraction")
plt.title("Calibration curve — test set"); plt.legend(); plt.tight_layout(); plt.show()

importance = pd.Series(cat_model.feature_importances_, index=feature_columns).sort_values(ascending=False)
display(importance.to_frame("catboost_feature_importance"))
importance.head(20).sort_values().plot(kind="barh", figsize=(8, 6))
plt.title("Top CatBoost feature importances"); plt.xlabel("Importance")
plt.tight_layout(); plt.show()

## 10. Optional source-held-out evaluation

When combining multiple datasets, a random split tests performance on rows drawn from the same mixture. It does not test generalization to a new source. Use the next cell to evaluate one dataset as an external holdout. It refits CatBoost on all other sources and tests on the held-out source; feature columns must be harmonized as above.

In [ ]:
# Set to an exact filename in source.value_counts().index, e.g. "dataset_b.csv".
HOLDOUT_SOURCE = None

if HOLDOUT_SOURCE is not None:
    train_mask = source != HOLDOUT_SOURCE
    test_mask = source == HOLDOUT_SOURCE
    if not test_mask.any() or not train_mask.any():
        raise ValueError("HOLDOUT_SOURCE must name one loaded source and leave at least one source for training.")
    X_src_train, y_src_train = X.loc[train_mask], y.loc[train_mask]
    X_src_test, y_src_test = X.loc[test_mask], y.loc[test_mask]
    if y_src_train.nunique() < 2:
        raise ValueError("Training sources need both target classes.")
    source_model = CatBoostClassifier(
        iterations=2500, depth=7, learning_rate=0.035,
        loss_function="Logloss", eval_metric="AUC",
        random_seed=RANDOM_STATE, l2_leaf_reg=5,
        verbose=False, allow_writing_files=False
    )
    source_model.fit(X_src_train, y_src_train, verbose=False)
    src_prob = source_model.predict_proba(X_src_test)[:, 1]
    src_pred = (src_prob >= chosen_threshold).astype(int)
    print("Held-out source:", HOLDOUT_SOURCE)
    print("Rows:", len(y_src_test), "prevalence:", round(float(y_src_test.mean()), 4))
    print("ROC-AUC:", round(roc_auc_score(y_src_test, src_prob), 4) if y_src_test.nunique() == 2 else "undefined: one class only")
    print(classification_report(y_src_test, src_pred, target_names=["No diabetes", "Diabetes"], zero_division=0))

## 11. Export model artifact and schema

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# CatBoost native format for deployment, plus metadata for backend validation.
cat_model.save_model(str(ARTIFACT_DIR / "diabetes_catboost.cbm"))
metadata = {
    "model_name": "CatBoostClassifier",
    "target": TARGET,
    "positive_class": 1,
    "threshold": chosen_threshold,
    "feature_names": feature_columns,
    "dataset_sources": sorted(source.unique().tolist()),
    "metrics": metrics,
    "warning": (
        "Educational model, not a diagnostic. Probability calibration and "
        "population prevalence must be validated on representative data."
    )
}
with open(ARTIFACT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=float)

print("Exported:", ARTIFACT_DIR / "diabetes_catboost.cbm")
print("Exported:", ARTIFACT_DIR / "model_metadata.json")

## 12. Import website suggestions

The Suggestions page submits entries to a shared backend. To have this notebook fetch the same live collection, set `DIABETES_SUGGESTIONS_API_URL` to the backend JSON export endpoint (for example `https://YOUR-SERVICE.example.com/api/suggestions/export.json`) before running the code cell below. If that variable is unset or the endpoint is unreachable, the cell falls back to a local `suggestions.json` file. It compiles the entries into `suggestions_compiled.md`. Suggestions are product feedback, not training labels, and must **never** be added to diabetes training data as if they were clinical observations.


In [ ]:
from pathlib import Path
import json
import os
from datetime import datetime, timezone
from urllib.request import Request, urlopen

SUGGESTIONS_JSON = Path("suggestions.json")
SUGGESTIONS_MD = Path("suggestions_compiled.md")
# Optional shared-backend sync. Set this environment variable to the backend's
# JSON export endpoint, e.g. https://YOUR-SERVICE.example.com/api/suggestions/export.json
SUGGESTIONS_API_URL = os.getenv("DIABETES_SUGGESTIONS_API_URL", "").strip()

payload = None
if SUGGESTIONS_API_URL:
    try:
        request = Request(SUGGESTIONS_API_URL, headers={"Accept": "application/json", "User-Agent": "GlucoGuide-Notebook/1.0"})
        with urlopen(request, timeout=20) as response:
            payload = json.loads(response.read().decode("utf-8"))
        print(f"Fetched shared suggestions from {SUGGESTIONS_API_URL}")
    except Exception as exc:
        print(f"Could not fetch shared suggestions ({exc}). Will try local suggestions.json instead.")

if payload is None and SUGGESTIONS_JSON.exists():
    payload = json.loads(SUGGESTIONS_JSON.read_text(encoding="utf-8"))

if payload is None:
    print("No suggestions available. Set DIABETES_SUGGESTIONS_API_URL to the shared backend export endpoint, or export suggestions.json from the website and place it in the notebook working directory.")
else:
    # Backend returns an object with a `suggestions` list; also accept a bare list.
    entries = payload if isinstance(payload, list) else payload.get("suggestions", [])
    if not isinstance(entries, list):
        raise ValueError("Expected a list of suggestion entries in the JSON payload")

    normalized = []
    for item in entries:
        if not isinstance(item, dict):
            continue
        suggestion = str(item.get("suggestion", "")).strip()
        if not suggestion:
            continue
        normalized.append({
            "id": str(item.get("id", "")),
            "createdAt": str(item.get("createdAt", "")),
            "category": str(item.get("category", "Other")),
            "suggestion": suggestion,
        })

    lines = [
        "# GlucoGuide Website Suggestions",
        "",
        f"Compiled: {datetime.now(timezone.utc).isoformat()}",
        f"Total suggestions: {len(normalized)}",
        "",
        "These entries are product feedback, not clinical observations or diabetes training labels. Review and triage them before making product or model changes.",
        "",
    ]
    if not normalized:
        lines.append("_No suggestions have been submitted._")
    for index, item in enumerate(normalized, start=1):
        lines.extend([
            f"## {index}. {item['category']}",
            "",
            f"- Submitted: {item['createdAt'] or 'unknown'}",
            f"- Reference: {item['id'] or 'not provided'}",
            "",
            item["suggestion"],
            "",
        ])
    SUGGESTIONS_MD.write_text("\n".join(lines), encoding="utf-8")
    print(f"Compiled {len(normalized)} suggestions into {SUGGESTIONS_MD.resolve()}")


## 13. Responsible-use and data compatibility checklist

- [ ] Confirm target definition is identical across sources (diagnosed diabetes vs diabetes/prediabetes are not interchangeable).
- [ ] Confirm feature units and coding are equivalent; use `COLUMN_MAPS` only for genuine semantic equivalence.
- [ ] Do not combine the balanced and natural-prevalence BRFSS files as independent sources; they are derived from the same underlying survey and can overlap.
- [ ] Do not combine duplicate/overlapping respondents; source-aware or group-aware splitting is preferable.
- [ ] Prefer a representative, naturally imbalanced dataset for probability and deployment assessment.
- [ ] Assess subgroup performance, calibration, missingness, data shift, privacy, and applicable regulatory requirements.
- [ ] Keep a held-out external dataset untouched for final generalization claims.
- [ ] Use clinical review before making screening decisions; this model is not a diagnosis.

**Interpretation:** CatBoost is the only algorithm in this notebook. Multiple datasets are supported through explicit schema harmonization and source-aware reporting. Better performance is not guaranteed until the notebook is run on the actual datasets.